# JupyterLite で学ぶ 日本語テキスト分析 入門チュートリアル（janome ＋ wordcloud）

このノートブックは、**JupyterLite（ブラウザだけで動く Jupyter 環境）** 上で、
純 Python の形態素解析器 **janome** と **wordcloud** を使って、日本語の文章を「数える・見える化する・比べる」
テキスト分析の基本を学ぶためのチュートリアルです。

## 対象者
- Python の基本（リスト・辞書・for 文・関数）を学んだ方
- ニュース記事・アンケートの自由回答・報告書などの日本語テキストを分析してみたい方
- 「形態素解析」「TF-IDF」「ワードクラウド」という言葉を実際に手を動かして理解したい方

## このチュートリアルで学ぶこと
0. 環境準備（JupyterLite 用）
1. 日本語テキスト分析と形態素解析
2. janome の基本（単語に分ける・品詞を調べる）
3. 品詞によるフィルタとストップワード
4. 分析用コーパスの準備
5. 単語の頻度分析
6. ワードクラウド
7. TF-IDF（文書を特徴づける単語）
8. 文書間の類似度
9. 共起ネットワーク
10. 辞書による簡易極性分析
11. まとめと総合演習

## 使い方
- セルを上から順に `Shift + Enter` で実行してください。途中を飛ばすと変数が未定義になります。
- 各章の最後に **練習問題** があります。「解答欄」に自分で書いてから「解答例」を開いて確認しましょう。

---
## 0. 環境準備（JupyterLite 用）

- **janome**：純 Python の形態素解析器。辞書が同梱されているので、インストールするだけで日本語を単語に分けられます。
- **wordcloud**：Pyodide に同梱されています。日本語を描くには日本語フォントのファイルが必要で、
  `japanize-matplotlib-jlite` に含まれる IPAex ゴシックのパスを `get_font_ttf_path()` で取得して渡します。
- **scikit-learn**（TF-IDF）、**networkx**（共起ネットワーク）も Pyodide に同梱されています。

In [ ]:
# JupyterLite 用のパッケージインストール（初回は数十秒かかることがあります）
try:
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "scikit-learn", "networkx", "wordcloud", "janome", "japanize-matplotlib-jlite"])
    print("piplite でのインストールが完了しました")
except ImportError:
    print("piplite がない環境（ローカルの Jupyter）なのでスキップしました")

In [ ]:
from collections import Counter
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite  # 日本語フォント（pyplot の後に import）
from janome.tokenizer import Tokenizer
from wordcloud import WordCloud
import networkx as nx
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

FONT_PATH = japanize_matplotlib_jlite.get_font_ttf_path()   # ワードクラウド用の日本語フォント
print("日本語フォント:", FONT_PATH)

---
## 1. 日本語テキスト分析と形態素解析

英語の文章は空白で区切れば単語になりますが、日本語には空白がありません。
そこで、文章を **意味を持つ最小の単位（形態素）** に分割し、それぞれの **品詞** を判定する処理が必要になります。
これが **形態素解析** です。

```
経済学は希少な資源の配分を研究する学問です
→ 経済 / 学 / は / 希少 / な / 資源 / の / 配分 / を / 研究 / する / 学問 / です
```

| ツール | 特徴 |
|---|---|
| MeCab | 高速・高精度。C++ 製で別途インストールが必要（JupyterLite では使えない） |
| **janome** | 純 Python。辞書同梱でインストールだけで動く。小〜中規模のテキストなら十分 |
| SudachiPy | 分割単位を選べる。C 拡張が必要 |

テキスト分析の基本的な流れは次のとおりです。

1. 形態素解析で単語に分ける
2. 必要な品詞だけ残し、意味の薄い語（ストップワード）を除く
3. 数える（頻度、TF-IDF）
4. 見える化する（棒グラフ、ワードクラウド、ネットワーク）
5. 比べる（文書間の類似度、グループ間の違い）

---
## 2. janome の基本

### 2.1 文章を単語に分ける

`Tokenizer()` を作り、`tokenize()` に文章を渡すと、単語（トークン）が 1 つずつ返ってきます。
各トークンは次の属性を持っています。

| 属性 | 意味 | 例（「研究する」の「する」） |
|---|---|---|
| `surface` | 表層形（文章に現れた形） | する |
| `part_of_speech` | 品詞（カンマ区切りで細分類まで） | 動詞,自立,*,* |
| `base_form` | 原形（辞書に載っている形） | する |
| `reading` | 読み | スル |

In [ ]:
tokenizer = Tokenizer()

text = "経済学は希少な資源の配分を研究する学問です。"
for token in tokenizer.tokenize(text):
    print(f"{token.surface:<6} | {token.part_of_speech:<20} | 原形: {token.base_form:<6} | 読み: {token.reading}")

### 2.2 分かち書き

品詞がいらず「単語のリスト」だけ欲しいときは `wakati=True` を指定します。

In [ ]:
words = list(tokenizer.tokenize("名古屋で経済学を学ぶ学生が増えている", wakati=True))
print(words)
print("単語数:", len(words))

### 2.3 表層形と原形

「増えている」の「増え」は動詞「増える」の活用形です。単語を数えるときは、活用形をそろえるために **原形（`base_form`）** を使うのが基本です。

In [ ]:
for token in tokenizer.tokenize("物価が上がり、消費が減った。企業の投資も減っている。"):
    pos = token.part_of_speech.split(",")[0]
    if pos == "動詞":
        print(f"表層形: {token.surface:<4} → 原形: {token.base_form}")

### 練習問題 1

1. 「日本銀行は金融政策決定会合で政策金利を据え置いた」を形態素解析し、表層形と品詞（大分類だけ）を表示してください。
2. 同じ文を分かち書きして、単語数を数えてください。
3. 「働く」「働き」「働いた」「働けば」を含む文を作り、動詞の原形がすべて「働く」になることを確認してください。

In [ ]:
# 練習問題 1 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 1 の解答例を見る</strong></summary>

```python
# 1
for token in tokenizer.tokenize("日本銀行は金融政策決定会合で政策金利を据え置いた"):
    print(token.surface, token.part_of_speech.split(",")[0])

# 2
print(len(list(tokenizer.tokenize("日本銀行は金融政策決定会合で政策金利を据え置いた", wakati=True))))

# 3
for token in tokenizer.tokenize("毎日働く。昨日も働き、一昨日も働いた。働けば給料がもらえる。"):
    if token.part_of_speech.startswith("動詞"):
        print(token.surface, "→", token.base_form)
```

</details>

---
## 3. 品詞によるフィルタとストップワード

「は」「の」「です」のような助詞・助動詞は、どの文章にも出てくるので分析の役に立ちません。
通常は **名詞・動詞・形容詞** だけを残します。さらに、「する」「ある」「こと」のように残しても意味の薄い語を
**ストップワード** として除きます。

この処理を関数 `extract_words()` にまとめておくと、以降の章で何度も使えます。

In [ ]:
STOPWORDS = {
    "する", "ある", "なる", "いる", "おる", "いく", "くる", "みる", "しまう", "できる", "れる", "られる", "せる", "いう", "思う",
    "こと", "もの", "ため", "よう", "これ", "それ", "あれ", "の", "ん", "さ", "そう",
    "年", "月", "日", "円", "％", "%", "人", "つ", "中", "上", "下", "前", "後", "的",
}

def extract_words(text, pos_keep=("名詞", "動詞", "形容詞"), stopwords=STOPWORDS):
    """文章から、指定した品詞の単語（原形）を取り出してリストで返す。"""
    words = []
    for token in tokenizer.tokenize(text):
        pos = token.part_of_speech.split(",")
        if pos[0] not in pos_keep:
            continue
        if pos[0] == "名詞" and pos[1] in ("数", "非自立", "代名詞", "接尾"):
            continue                       # 数字・「こと」などの非自立名詞・代名詞・接尾辞は除く
        word = token.base_form if token.base_form != "*" else token.surface
        if word in stopwords or len(word) < 2:
            continue
        words.append(word)
    return words

sample = "日本銀行は金融政策決定会合で政策金利を据え置いたが、物価の上昇が続けば追加の利上げを検討する。"
print(extract_words(sample))
print("名詞だけ:", extract_words(sample, pos_keep=("名詞",)))

### 複合名詞をまとめる

「金融政策決定会合」は 4 つの名詞に分かれますが、1 つの用語として扱いたいこともあります。
janome の `Analyzer` に `CompoundNounFilter` を組み合わせると、連続する名詞を 1 語にまとめられます。

In [ ]:
from janome.analyzer import Analyzer
from janome.tokenfilter import CompoundNounFilter, POSKeepFilter
from janome.charfilter import UnicodeNormalizeCharFilter

analyzer = Analyzer(
    char_filters=[UnicodeNormalizeCharFilter()],          # 全角英数を半角に、などの正規化
    tokenizer=tokenizer,
    token_filters=[CompoundNounFilter(), POSKeepFilter(["名詞"])],
)
print([token.surface for token in analyzer.analyze(sample)])

### 練習問題 2

1. `extract_words()` を使って「企業の設備投資が増加し、雇用も改善している。しかし中小企業では人手不足が深刻だ。」から名詞・動詞・形容詞を取り出してください。
2. 同じ文から **形容詞だけ** を取り出してください（ヒント：`pos_keep=("形容詞",)`）。
3. ストップワードに「企業」を追加した集合を作り、1 の結果から「企業」が消えることを確認してください。

In [ ]:
# 練習問題 2 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 2 の解答例を見る</strong></summary>

```python
s = "企業の設備投資が増加し、雇用も改善している。しかし中小企業では人手不足が深刻だ。"

# 1
print(extract_words(s))

# 2
print(extract_words(s, pos_keep=("形容詞",)))

# 3
my_stop = STOPWORDS | {"企業"}
print(extract_words(s, stopwords=my_stop))
```

</details>

---
## 4. 分析用コーパスの準備

分析対象の文書の集まりを **コーパス** と呼びます。ここでは、経済ニュース風に書いた架空の短い記事 6 本を使います
（実際の記事ではありません）。実務では CSV ファイルなどから読み込みますが、流れは同じです。

In [ ]:
corpus = {
    "金融政策": "日本銀行は金融政策決定会合で政策金利の据え置きを決めた。物価上昇率は目標の2%を上回っており、"
              "市場では年内の追加利上げを予想する声が強まっている。長期金利は小幅に上昇した。",
    "物価動向": "消費者物価指数は前年比で3%上昇し、食料品と電気料金の値上がりが家計を圧迫している。"
              "政府は物価高対策として補助金の延長を検討しているが、財政負担の増加を懸念する意見もある。",
    "雇用情勢": "有効求人倍率は高水準を維持し、人手不足が深刻化している。企業は賃金を引き上げて人材を確保しようとしており、"
              "春闘では大企業を中心に高い賃上げ率が実現した。中小企業の賃上げが課題である。",
    "貿易収支": "輸出は自動車と半導体製造装置が好調で増加したが、原油価格の上昇で輸入も増え、貿易収支は赤字が続いた。"
              "円安は輸出企業の収益を押し上げる一方、輸入物価を通じて家計の負担を増やしている。",
    "企業業績": "上場企業の決算は円安と価格転嫁の進展で増益となった。設備投資は過去最高水準に達し、"
              "特に半導体や電気自動車向けの投資が拡大している。ただし海外経済の減速が収益の下振れリスクとなっている。",
    "地域経済": "地方では人口減少と高齢化が進み、地域経済の縮小が懸念されている。観光需要の回復で宿泊業は改善したが、"
              "製造業の工場閉鎖が相次ぐ地域もある。自治体は企業誘致と移住促進に力を入れている。",
}
docs = pd.DataFrame({"title": list(corpus.keys()), "text": list(corpus.values())})
docs["length"] = docs["text"].str.len()
print(docs[["title", "length"]])

In [ ]:
# 各文書を単語リストに変換して列として保持しておく
docs["words"] = docs["text"].apply(extract_words)
docs["n_words"] = docs["words"].apply(len)
print(docs[["title", "n_words"]])
print("\n例（金融政策）:", docs.loc[0, "words"])

---
## 5. 単語の頻度分析

### 5.1 コーパス全体で数える

`collections.Counter` に単語リストを渡すと、単語ごとの出現回数を数えてくれます。

In [ ]:
all_words = list(itertools.chain.from_iterable(docs["words"]))
counter = Counter(all_words)
print("総単語数:", len(all_words), " 異なり語数:", len(counter))
print("\n上位 15 語:")
for word, count in counter.most_common(15):
    print(f"{word:<8} {count}")

In [ ]:
# 上位 20 語を横棒グラフに
top = pd.DataFrame(counter.most_common(20), columns=["word", "count"]).sort_values("count")
plt.figure(figsize=(7, 6))
plt.barh(top["word"], top["count"], color="steelblue")
plt.xlabel("出現回数")
plt.title("頻出語トップ 20")
plt.tight_layout()
plt.show()

### 5.2 文書ごとに数える

文書ごとの頻出語を見ると、それぞれの記事が何について書かれているかが分かります。

In [ ]:
for _, row in docs.iterrows():
    top5 = Counter(row["words"]).most_common(5)
    print(f"{row['title']}: {', '.join(f'{w}({c})' for w, c in top5)}")

### 5.3 文書・単語行列（document-term matrix）

「行 = 文書、列 = 単語、値 = 出現回数」の行列を作ると、機械的に比較や計算ができるようになります。
scikit-learn の `CountVectorizer` に、janome を使った分かち書き関数を `tokenizer` として渡します
（`token_pattern=None` を付けるのは、既定の英語用パターンを使わないためです）。

In [ ]:
count_vec = CountVectorizer(tokenizer=extract_words, token_pattern=None)
X_count = count_vec.fit_transform(docs["text"])
dtm = pd.DataFrame(X_count.toarray(), index=docs["title"], columns=count_vec.get_feature_names_out())
print("行列の形（文書数 × 語彙数）:", dtm.shape)
print(dtm[["物価", "企業", "投資", "金利", "地域", "賃上げ"]])

### 5.4 連続する 2 語（バイグラム）を数える

「人手 不足」「設備 投資」のように、**連続して現れる 2 語の組（バイグラム）** を数えると、単語単体より意味のまとまりが見えてきます。

In [ ]:
bigram_counter = Counter()
for text in docs["text"]:
    for sentence in text.split("。"):
        nouns_in_sentence = extract_words(sentence, pos_keep=("名詞",))
        bigram_counter.update(zip(nouns_in_sentence, nouns_in_sentence[1:]))   # 隣り合う名詞のペア

print("連続する名詞ペアの上位 10:")
for (w1, w2), cnt in bigram_counter.most_common(10):
    print(f"  {w1}{w2}: {cnt}")

### 練習問題 3

1. コーパス全体で **名詞だけ** を数え、上位 10 語を表示してください。
2. 「企業業績」の文書に出てくる単語のうち、コーパス全体で 1 回しか出てこない単語を表示してください（ヒント：`counter[word] == 1`）。
3. `dtm` を使って、各文書の総単語数（行の合計）を求めてください。

In [ ]:
# 練習問題 3 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 3 の解答例を見る</strong></summary>

```python
# 1
nouns = list(itertools.chain.from_iterable(docs["text"].apply(lambda t: extract_words(t, pos_keep=("名詞",)))))
print(Counter(nouns).most_common(10))

# 2
biz_words = docs.loc[docs["title"] == "企業業績", "words"].iloc[0]
print([w for w in set(biz_words) if counter[w] == 1])

# 3
print(dtm.sum(axis=1))
```

</details>

---
## 6. ワードクラウド

**ワードクラウド** は、頻度の高い単語ほど大きく描く可視化です。`generate_from_frequencies()` に
`{単語: 回数}` の辞書を渡します。日本語を描くには **`font_path` に日本語フォントのファイル** を指定する必要があります。

In [ ]:
wc = WordCloud(
    font_path=FONT_PATH,          # 日本語フォント（必須）
    width=800, height=400,
    background_color="white",
    colormap="viridis",
    max_words=60,
).generate_from_frequencies(counter)

plt.figure(figsize=(10, 5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("経済ニュース コーパスのワードクラウド")
plt.show()

文書ごとに描くと、テーマの違いが一目で分かります。

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (_, row) in zip(axes.ravel(), docs.iterrows()):
    wc_doc = WordCloud(font_path=FONT_PATH, width=400, height=250, background_color="white",
                       colormap="tab10", max_words=30).generate_from_frequencies(Counter(row["words"]))
    ax.imshow(wc_doc, interpolation="bilinear")
    ax.set_title(row["title"])
    ax.axis("off")
plt.tight_layout()
plt.show()

ワードクラウドは「印象をつかむ」には便利ですが、単語の位置や色に意味はありません。
報告書では、必ず頻度表や棒グラフなどの **数値の根拠** と一緒に示しましょう。

### 練習問題 4

1. 名詞だけのワードクラウドを描いてください。
2. `colormap="Reds"`、`background_color="black"` にして雰囲気を変えてみてください。
3. 「雇用情勢」と「地域経済」の 2 文書を合わせた単語の頻度でワードクラウドを描いてください。

In [ ]:
# 練習問題 4 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 4 の解答例を見る</strong></summary>

```python
# 1
noun_counter = Counter(nouns) if "nouns" in dir() else Counter(
    itertools.chain.from_iterable(docs["text"].apply(lambda t: extract_words(t, pos_keep=("名詞",)))))
wc1 = WordCloud(font_path=FONT_PATH, width=800, height=400, background_color="white").generate_from_frequencies(noun_counter)
plt.figure(figsize=(10, 5)); plt.imshow(wc1); plt.axis("off"); plt.show()

# 2
wc2 = WordCloud(font_path=FONT_PATH, width=800, height=400, background_color="black", colormap="Reds").generate_from_frequencies(counter)
plt.figure(figsize=(10, 5)); plt.imshow(wc2); plt.axis("off"); plt.show()

# 3
sub = docs[docs["title"].isin(["雇用情勢", "地域経済"])]
sub_counter = Counter(itertools.chain.from_iterable(sub["words"]))
wc3 = WordCloud(font_path=FONT_PATH, width=800, height=400, background_color="white").generate_from_frequencies(sub_counter)
plt.figure(figsize=(10, 5)); plt.imshow(wc3); plt.axis("off"); plt.show()
```

</details>

---
## 7. TF-IDF（文書を特徴づける単語）

単純な頻度では、「企業」「増加」のようにどの文書にも出てくる語が上位に来てしまいます。
**TF-IDF** は「その文書での頻度（TF）」に「その語が現れる文書の少なさ（IDF）」を掛けた重みで、
**その文書に特徴的な語** を高く評価します。

$$
\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \log\frac{N}{\text{DF}(t)}
$$

- $\text{TF}(t, d)$：文書 $d$ における語 $t$ の出現回数（の割合）
- $N$：文書数、$\text{DF}(t)$：語 $t$ を含む文書の数

scikit-learn の `TfidfVectorizer` を使えば 1 行で計算できます。

In [ ]:
tfidf_vec = TfidfVectorizer(tokenizer=extract_words, token_pattern=None)
X_tfidf = tfidf_vec.fit_transform(docs["text"])
tfidf = pd.DataFrame(X_tfidf.toarray(), index=docs["title"], columns=tfidf_vec.get_feature_names_out())
print("行列の形:", tfidf.shape)

# 文書ごとに TF-IDF が高い語トップ 5
for title, row in tfidf.iterrows():
    top5 = row.sort_values(ascending=False).head(5)
    print(f"{title}: {', '.join(f'{w}({v:.2f})' for w, v in top5.items())}")

In [ ]:
# 頻度と TF-IDF の違いを比べる：「企業」はどの文書にもあるので TF-IDF では低くなる
compare = pd.DataFrame({
    "出現回数（全体）": [counter[w] for w in ["企業", "物価", "半導体", "観光"]],
    "出現文書数": [(dtm[w] > 0).sum() for w in ["企業", "物価", "半導体", "観光"]],
    "TF-IDF の最大値": [tfidf[w].max() for w in ["企業", "物価", "半導体", "観光"]],
}, index=["企業", "物価", "半導体", "観光"])
print(compare.round(3))

文書ごとの特徴語トップ 3 を 1 つの表にまとめると、コーパス全体の見取り図になります。

In [ ]:
summary_rows = []
for title, row in tfidf.iterrows():
    top3 = row.sort_values(ascending=False).head(3)
    summary_rows.append({"文書": title, "特徴語 1": top3.index[0], "特徴語 2": top3.index[1], "特徴語 3": top3.index[2],
                         "単語数": int(dtm.loc[title].sum())})
print(pd.DataFrame(summary_rows).to_string(index=False))

### 練習問題 5

1. 「地域経済」の文書について、TF-IDF が高い語トップ 10 を棒グラフにしてください。
2. コーパス全体で TF-IDF の合計が最も大きい語トップ 10 を求めてください。
3. `TfidfVectorizer(..., min_df=2)` として「2 文書以上に出てくる語」だけで行列を作り、語彙数がどれだけ減るか確認してください。

In [ ]:
# 練習問題 5 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 5 の解答例を見る</strong></summary>

```python
# 1
top10 = tfidf.loc["地域経済"].sort_values(ascending=False).head(10).sort_values()
plt.figure(figsize=(6, 4))
plt.barh(top10.index, top10.values, color="darkorange")
plt.title("「地域経済」の特徴語（TF-IDF）")
plt.show()

# 2
print(tfidf.sum().sort_values(ascending=False).head(10).round(3))

# 3
vec2 = TfidfVectorizer(tokenizer=extract_words, token_pattern=None, min_df=2)
X2 = vec2.fit_transform(docs["text"])
print("語彙数:", tfidf.shape[1], "→", X2.shape[1])
```

</details>

---
## 8. 文書間の類似度

TF-IDF ベクトルどうしの **コサイン類似度**（ベクトルのなす角のコサイン、1 に近いほど似ている）を計算すると、
「どの文書とどの文書が似ているか」を数値で比べられます。

In [ ]:
sim = pd.DataFrame(cosine_similarity(X_tfidf), index=docs["title"], columns=docs["title"])
print(sim.round(2))

In [ ]:
# ヒートマップで見る
plt.figure(figsize=(6, 5))
plt.imshow(sim.values, cmap="Blues", vmin=0, vmax=1)
plt.colorbar(label="コサイン類似度")
plt.xticks(range(len(sim)), sim.columns, rotation=45)
plt.yticks(range(len(sim)), sim.index)
for i in range(len(sim)):
    for j in range(len(sim)):
        plt.text(j, i, f"{sim.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if sim.values[i, j] > 0.5 else "black")
plt.title("文書間の類似度")
plt.tight_layout()
plt.show()

In [ ]:
# 最も似ている文書のペアを探す（対角線は除く）
pairs = []
for i, j in itertools.combinations(range(len(sim)), 2):
    pairs.append((sim.index[i], sim.columns[j], sim.values[i, j]))
pairs_df = pd.DataFrame(pairs, columns=["文書 A", "文書 B", "類似度"]).sort_values("類似度", ascending=False)
print(pairs_df.round(3).head())

### 簡単な文書検索

同じ仕組みで、質問文に最も近い文書を探す **検索** ができます。質問文を同じ `TfidfVectorizer` でベクトルにして、
各文書との類似度を計算するだけです。

In [ ]:
def search(query, top_n=3):
    q_vec = tfidf_vec.transform([query])
    scores = cosine_similarity(q_vec, X_tfidf)[0]
    ranking = pd.Series(scores, index=docs["title"]).sort_values(ascending=False)
    return ranking.head(top_n)

print(search("賃金の引き上げと人手不足"))
print()
print(search("輸出と円安の影響"))

### 類似度をもとに文書をグループ分けする

類似度（距離）をもとに文書を **階層的クラスタリング** でまとめると、テーマの近い文書のグループが得られます。
文書が多いときに全体を俯瞰するのに便利です（`scipy` は Pyodide に同梱されています）。

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

distance = 1 - sim.values[np.triu_indices(len(sim), k=1)]      # 類似度 → 距離（上三角だけ取り出す）
Z = linkage(distance, method="average")

plt.figure(figsize=(7, 4))
dendrogram(Z, labels=list(sim.index), leaf_font_size=11)
plt.ylabel("距離（1 − 類似度）")
plt.title("文書の階層的クラスタリング")
plt.show()

clusters = fcluster(Z, t=3, criterion="maxclust")             # 3 グループに分ける
print(pd.Series(clusters, index=sim.index, name="cluster"))

---
## 9. 共起ネットワーク

同じ文の中に一緒に現れる（**共起** する）単語のペアを数え、ネットワークとして描くと、
「どの語とどの語が結びついて語られているか」が見えてきます。

1. 文書を「。」で文に分ける
2. 各文の単語の組み合わせ（`itertools.combinations`）を数える
3. 回数の多いペアを辺とするグラフを `networkx` で描く

In [ ]:
noun_counter = Counter(itertools.chain.from_iterable(docs["text"].apply(lambda t: extract_words(t, pos_keep=("名詞",)))))
top_nouns = {w for w, _ in noun_counter.most_common(30)}      # 頻出名詞 30 語に限定して見やすくする

pair_counter = Counter()
for text in docs["text"]:
    for sentence in text.split("。"):
        words_in_sentence = sorted(set(extract_words(sentence, pos_keep=("名詞",))) & top_nouns)
        for pair in itertools.combinations(words_in_sentence, 2):
            pair_counter[pair] += 1

print("共起ペア数:", len(pair_counter))
print("上位 10 ペア:")
for pair, cnt in pair_counter.most_common(10):
    print(f"  {pair[0]} — {pair[1]}: {cnt}")

In [ ]:
G = nx.Graph()
for (w1, w2), cnt in pair_counter.items():
    G.add_edge(w1, w2, weight=cnt)
print("ノード数:", G.number_of_nodes(), " エッジ数:", G.number_of_edges())

pos = nx.spring_layout(G, k=0.9, seed=0)
plt.figure(figsize=(11, 9))
nx.draw_networkx_nodes(G, pos, node_size=[200 + 250 * noun_counter[n] for n in G.nodes()], node_color="lightyellow", edgecolors="gray")
nx.draw_networkx_edges(G, pos, width=[1.5 * G[u][v]["weight"] for u, v in G.edges()], alpha=0.4)
nx.draw_networkx_labels(G, pos, font_size=11, font_family="IPAexGothic")
plt.title("頻出名詞の共起ネットワーク（同じ文に一緒に出た語を線で結ぶ）")
plt.axis("off")
plt.show()

In [ ]:
# 次数中心性：多くの語とつながっている「ハブ」となる語
centrality = pd.Series(nx.degree_centrality(G)).sort_values(ascending=False)
print(centrality.head(8).round(3))

### 練習問題 6

1. 共起回数が 2 回以上のペアだけでネットワークを作り直し、ノード数とエッジ数を表示してください（描画はしなくて構いません）。
2. 「企業」と共起している語をすべて表示してください（ヒント：`G.neighbors("企業")`、または `pair_counter` を走査）。
3. `nx.betweenness_centrality(G)` で媒介中心性を計算し、上位 5 語を表示してください。

In [ ]:
# 練習問題 6 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 6 の解答例を見る</strong></summary>

```python
# 1
G1 = nx.Graph()
for (w1, w2), cnt in pair_counter.items():
    if cnt >= 2:
        G1.add_edge(w1, w2, weight=cnt)
print(G1.number_of_nodes(), G1.number_of_edges())

# 2
print(sorted(G.neighbors("企業")) if "企業" in G else "（企業はネットワークに含まれていません）")

# 3
print(pd.Series(nx.betweenness_centrality(G)).sort_values(ascending=False).head(5).round(3))
```

</details>

---
## 10. 辞書による簡易極性分析

文章が「ポジティブか、ネガティブか」を数値化するのが **極性分析（感情分析）** です。
最も簡単な方法は、あらかじめ用意した **極性辞書**（単語 → +1 / −1）で単語を照合し、スコアを合計することです。

ここでは経済ニュース向けの小さな辞書を自分で定義します。実務では公開されている極性辞書（日本語評価極性辞書など）を使います。

In [ ]:
POLARITY = {
    # ポジティブ
    "増加": 1, "上昇": 1, "拡大": 1, "改善": 1, "好調": 1, "回復": 1, "増益": 1, "実現": 1, "確保": 1, "高水準": 1,
    # ネガティブ
    "減少": -1, "下落": -1, "縮小": -1, "悪化": -1, "低迷": -1, "懸念": -1, "赤字": -1, "圧迫": -1, "不足": -1,
    "深刻": -1, "減速": -1, "リスク": -1, "閉鎖": -1, "負担": -1, "課題": -1,
}

def polarity_score(words):
    """単語リストの極性スコア（合計 / 単語数）と、ヒットした語を返す。"""
    hits = [(w, POLARITY[w]) for w in words if w in POLARITY]
    total = sum(v for _, v in hits)
    return total / max(len(words), 1), hits

results = []
for _, row in docs.iterrows():
    score, hits = polarity_score(row["words"])
    results.append({"title": row["title"], "score": round(score, 3),
                    "positive": [w for w, v in hits if v > 0], "negative": [w for w, v in hits if v < 0]})
polarity_df = pd.DataFrame(results)
print(polarity_df[["title", "score"]])
print("\n例（企業業績）: +", polarity_df.loc[4, "positive"], " −", polarity_df.loc[4, "negative"])

In [ ]:
plt.figure(figsize=(7, 4))
colors = ["tomato" if s < 0 else "seagreen" for s in polarity_df["score"]]
plt.bar(polarity_df["title"], polarity_df["score"], color=colors)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("極性スコア（単語あたり）")
plt.title("文書ごとの極性スコア")
plt.show()

辞書法は簡単で解釈しやすい反面、**否定（「改善しなかった」）や文脈（「懸念が後退」）を扱えない** という限界があります。
より高度な分析には機械学習（ラベル付きデータで分類器を学習）が使われますが、まずは辞書法で全体の傾向をつかむのが第一歩です。

### 否定表現の簡易処理

「改善しなかった」のように、極性語の直後に **否定** が続く場合は符号を反転させる、という簡単な工夫だけでも精度が上がります。
janome のトークン列を順番に見て、極性語の **直後 3 トークン以内** に「ない」「ず」「なかっ」などの否定が現れたら符号を反転します
（「改善し**なかっ**た」「赤字では**ない**」のように、極性語と否定の間に「し」「で」「は」が挟まることが多いためです）。

In [ ]:
NEGATIONS = {"ない", "ず", "ぬ", "なかっ", "なく"}

def polarity_with_negation(text):
    tokens = list(tokenizer.tokenize(text))
    score, hits = 0, []
    for i, token in enumerate(tokens):
        word = token.base_form if token.base_form != "*" else token.surface
        if word in POLARITY:
            value = POLARITY[word]
            following = [t.surface for t in tokens[i + 1:i + 4]]        # 直後 3 トークン
            if any(w in NEGATIONS for w in following):
                value = -value
                word = word + "（否定）"
            score += value
            hits.append((word, value))
    return score, hits

for sentence in ["業績は改善した。", "業績は改善しなかった。", "懸念は残るが赤字ではない。"]:
    print(sentence, "→", polarity_with_negation(sentence))

### 練習問題 7

1. `POLARITY` 辞書に「据え置き: 0」「利上げ: -1」「値上がり: -1」「賃上げ: +1」を追加して、各文書のスコアを計算し直してください。
2. 文書を「。」で文に分け、文ごとの極性スコアを計算して、最もネガティブな文を表示してください。

In [ ]:
# 練習問題 7 の解答欄：ここにコードを書いてください

<details>
<summary><strong>練習問題 7 の解答例を見る</strong></summary>

```python
# 1
POLARITY.update({"据え置き": 0, "利上げ": -1, "値上がり": -1, "賃上げ": 1})
for _, row in docs.iterrows():
    score, _ = polarity_score(row["words"])
    print(row["title"], round(score, 3))

# 2
sentences = []
for text in docs["text"]:
    for s in text.split("。"):
        if s:
            score, _ = polarity_score(extract_words(s))
            sentences.append((score, s))
print(min(sentences))
```

</details>

---
## 11. まとめ

| ステップ | 道具 | 主な関数 |
|---|---|---|
| 形態素解析 | janome | `Tokenizer().tokenize()`、`surface` / `part_of_speech` / `base_form` |
| 前処理 | 自作関数 | 品詞フィルタ、ストップワード、`Analyzer` + `CompoundNounFilter` |
| 頻度分析 | collections / scikit-learn | `Counter`、`CountVectorizer(tokenizer=..., token_pattern=None)` |
| 可視化 | wordcloud / matplotlib | `WordCloud(font_path=...).generate_from_frequencies()` |
| 特徴語 | scikit-learn | `TfidfVectorizer` |
| 類似度・検索 | scikit-learn | `cosine_similarity` |
| 共起ネットワーク | networkx | `itertools.combinations`、`nx.Graph`、`degree_centrality` |
| 極性分析 | 自作辞書 | 単語の極性の合計 |

## 次のステップ

- `python/networkx/networkx_beginner_tutorial.ipynb` — ネットワーク分析をより詳しく
- `python/sklearn/sklearn_beginner_tutorial.ipynb` — TF-IDF を特徴量にした文書分類へ
- 実データ（アンケートの自由回答、議事録、有価証券報告書のテキスト）を CSV で用意して、同じ流れを試してみましょう

---
## 総合演習：顧客アンケートの自由回答分析

あるカフェチェーンの顧客アンケート（満足度 1〜5 と自由回答）を分析します。次のセルで架空の回答 12 件を作ります。

1. 回答ごとに名詞・形容詞を取り出し、コーパス全体の頻出語トップ 10 を表示してください。
2. 満足度が高いグループ（4 以上）と低いグループ（2 以下）に分け、それぞれのワードクラウドを並べて描いてください。
3. 各グループの TF-IDF 上位 5 語（そのグループを特徴づける語）を求めてください（ヒント：グループごとに回答を結合して 2 文書にする）。
4. 第 10 章の `POLARITY` に「おいしい: +1」「良い: +1」「丁寧: +1」「遅い: -1」「高い: -1」「狭い: -1」「悪い: -1」を追加し、回答ごとの極性スコアと満足度の相関係数を求めてください。

In [ ]:
# 総合演習の解答欄：ここにコードを書いてください
survey = pd.DataFrame({
    "satisfaction": [5, 4, 5, 2, 1, 4, 2, 5, 3, 1, 4, 2],
    "comment": [
        "コーヒーがおいしいし、店員の対応も丁寧で気持ちが良い。",
        "席が広くて落ち着ける。Wi-Fiが使えるので仕事にも便利。",
        "新しい季節限定メニューがおいしかった。また来たい。",
        "注文してから出てくるのが遅い。値段も高いと感じる。",
        "店内が狭くてうるさい。トイレが汚くて印象が悪い。",
        "駅から近くて便利。ケーキの種類が多くて良い。",
        "コーヒーの味は普通だが値段が高い。コスパが悪い。",
        "店員が親切で丁寧。落ち着いた雰囲気が良い。",
        "可もなく不可もなく。混雑していて席を探すのが大変だった。",
        "提供が遅いうえに注文を間違えられた。二度と行かない。",
        "朝のセットがお得でおいしい。通勤途中に寄りやすい。",
        "Wi-Fiが遅い。コンセントが少なくて不便。",
    ],
})
print(survey)

### 総合演習の解答例

自分で書いてから、次のセルを実行して結果を比べてみてください。

In [ ]:
# 1. 頻出語
survey["words"] = survey["comment"].apply(lambda t: extract_words(t, pos_keep=("名詞", "形容詞")))
survey_counter = Counter(itertools.chain.from_iterable(survey["words"]))
print(survey_counter.most_common(10))

# 2. 満足度グループ別ワードクラウド
high = survey[survey["satisfaction"] >= 4]
low = survey[survey["satisfaction"] <= 2]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (label, group) in zip(axes, [("満足度 4 以上", high), ("満足度 2 以下", low)]):
    c = Counter(itertools.chain.from_iterable(group["words"]))
    ax.imshow(WordCloud(font_path=FONT_PATH, width=500, height=300, background_color="white").generate_from_frequencies(c))
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()

# 3. グループごとの特徴語（2 文書として TF-IDF）
group_docs = ["".join(high["comment"]), "".join(low["comment"])]
vec = TfidfVectorizer(tokenizer=lambda t: extract_words(t, pos_keep=("名詞", "形容詞")), token_pattern=None)
X = vec.fit_transform(group_docs)
group_tfidf = pd.DataFrame(X.toarray(), index=["高満足", "低満足"], columns=vec.get_feature_names_out())
for name, row in group_tfidf.iterrows():
    print(name, ":", ", ".join(row.sort_values(ascending=False).head(5).index))

# 4. 極性スコアと満足度の相関
POLARITY.update({"おいしい": 1, "良い": 1, "丁寧": 1, "遅い": -1, "高い": -1, "狭い": -1, "悪い": -1})
survey["score"] = survey["words"].apply(lambda w: polarity_score(w)[0])
print(survey[["satisfaction", "score"]])
print("相関係数:", round(survey["satisfaction"].corr(survey["score"]), 3))

お疲れさまでした！ ここで学んだ「形態素解析 → 前処理 → 数える → 見える化 → 比べる」という流れは、
アンケート・議事録・報告書・SNS など、どんな日本語テキストにもそのまま応用できます。